In [ ]:
%%sql
SELECT DISTINCT
    e.is_dna,
    s.description AS activity_status_description,
    CASE
        WHEN e.is_dna = true THEN 'Did Not Attend'
        WHEN LOWER(TRIM(s.description)) LIKE '%cancel%' THEN 'Cancelled with greater than 24 hours notice'
        WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
    END AS derived_session_status_src_name
FROM silver_wip_activityentry e
LEFT JOIN silver_wip_activityheader h
    ON e.activity_header_id = h.id
LEFT JOIN silver_wip_activitystatus s
    ON h.activity_status_id = s.id
ORDER BY 1,2,3;

In [ ]:
%%sql
SELECT
    CASE
        WHEN e.is_dna = true THEN 'Did Not Attend'
        WHEN LOWER(TRIM(s.description)) LIKE '%cancel%' THEN 'Cancelled with greater than 24 hours notice'
        WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
    END AS derived_session_status_src_name,
    COUNT(*) AS cnt
FROM silver_wip_activityentry e
LEFT JOIN silver_wip_activityheader h
    ON e.activity_header_id = h.id
LEFT JOIN silver_wip_activitystatus s
    ON h.activity_status_id = s.id
GROUP BY
    CASE
        WHEN e.is_dna = true THEN 'Did Not Attend'
        WHEN LOWER(TRIM(s.description)) LIKE '%cancel%' THEN 'Cancelled with greater than 24 hours notice'
        WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
    END
ORDER BY cnt DESC;

In [ ]:
,
wip_source AS (
    -- WIP source values derived from activity entry, activity header and activity status
    -- current available WIP logic supports Attended and Did Not Attend for WIP001
    SELECT DISTINCT
        CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
        END AS session_status_src_name,

        LOWER(TRIM(
            CASE
                WHEN e.is_dna = true THEN 'Did Not Attend'
                WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
            END
        )) AS session_status_src_id,

        'WIP001' AS session_status_src_sys_inst_id
    FROM silver_wip_activityentry e
    LEFT JOIN silver_wip_activityheader h
        ON e.activity_header_id = h.id
    LEFT JOIN silver_wip_activitystatus st
        ON h.activity_status_id = st.id
    WHERE CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
          END IS NOT NULL
)

In [ ]:
FROM (
    SELECT * FROM mpb_source
    UNION
    SELECT * FROM wip_source
) s

In [ ]:
%%sql
SELECT *
FROM silver_rdm_session_status_add
ORDER BY session_status_src_sys_inst_id, session_status_src_name;

In [ ]:
%%sql
SELECT
    session_status_src_sys_inst_id,
    COUNT(*) AS total_rows
FROM silver_rdm_session_status_add
GROUP BY session_status_src_sys_inst_id
ORDER BY session_status_src_sys_inst_id;

In [ ]:
%%sql
SELECT
    session_status_src_sys_inst_id,
    COUNT(DISTINCT session_status_src_id) AS unique_src_id_count
FROM silver_rdm_session_status_add
GROUP BY session_status_src_sys_inst_id
ORDER BY session_status_src_sys_inst_id;

In [ ]:
%%sql
SELECT
    session_status_src_id,
    session_status_src_sys_inst_id,
    COUNT(*) AS cnt
FROM silver_rdm_session_status_add
GROUP BY session_status_src_id, session_status_src_sys_inst_id
HAVING COUNT(*) > 1;

In [ ]:
Implemented WIP session status logic in silver_rdm_session_status_add.

Created attributes:
- session_status_src_id
- session_status_src_name
- session_status_src_sys_inst_id

Source tables reviewed:
- silver_wip_activityentry
- silver_wip_activityheader
- silver_wip_activitystatus

Joins used:
- silver_wip_activityentry.activity_header_id = silver_wip_activityheader.id
- silver_wip_activityheader.activity_status_id = silver_wip_activitystatus.id

Implemented logic:
- is_dna = true -> Did Not Attend
- activity_date_time in the past -> Attended
- session_status_src_id derived using the same logic as session_status_src_name
- session_status_src_sys_inst_id = WIP001

Validation:
- total WIP rows loaded: 2
- unique src_id count: 2
- duplicate check returned no rows

Note:
The Monday definition also includes cancellation logic, but cancellation-specific source data was not confirmed in the reviewed WIP Silver tables. Based on the tested output from silver_wip_activityentry, silver_wip_activityheader, and silver_wip_activitystatus, only Attended and Did Not Attend were derivable at this stage. Cancellation logic will need to be revisited after checking additional WIP sources / Production availability.